# Arez — Devil's Advocate
**Model:** Mistral Small 3.1 24B · **Hardware:** A100 40GB

| Bước | Cell | Làm gì |
|---|---|---|
| 1 | **Cell 1** | Cài thư viện → **Restart Runtime** |
| 2 | **Cell 2** | Load model (~5 phút) |
| 3 | **Cell 3** | Sửa `USER_INPUT` → Run mỗi lần hỏi |
| 4 | **Cell 2b** | Reset lịch sử khi đổi chủ đề |

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 1 — CÀI THƯ VIỆN
# Install transformers từ GitHub main (có Mistral3Config)
# ⚠️ SAU KHI XONG: Runtime > Restart runtime
# ═══════════════════════════════════════════════════

import subprocess, sys

cmds = [
    # transformers từ GitHub main — có Mistral3Config
    [sys.executable, '-m', 'pip', 'install', '-q',
     'git+https://github.com/huggingface/transformers.git'],
    # bitsandbytes mới nhất
    [sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes>=0.46.1'],
    # accelerate
    [sys.executable, '-m', 'pip', 'install', '-q', 'accelerate>=0.27.0'],
]

for cmd in cmds:
    subprocess.check_call(cmd)

import transformers
print(f'✅ Xong. transformers: {transformers.__version__}')
print('⚠️  Bây giờ: Runtime > Restart runtime → chạy Cell 2')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2 — LOAD MODEL
# Chạy 1 lần sau khi restart. ~5 phút lần đầu.
# ═══════════════════════════════════════════════════

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'mistralai/Mistral-Small-3.1-24B-Instruct-2503'

# Kiểm tra GPU
assert torch.cuda.is_available(), (
    'Không tìm thấy GPU. Vào Runtime > Change runtime type > A100'
)
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} | VRAM: {vram:.0f} GB')

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print('Đang load tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print('Đang load model... (~5 phút lần đầu)')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.eval()

# System prompt
SYSTEM_PROMPT = (
    "Bạn là Arez — AI đóng vai Devil's Advocate.\n\n"
    "Nhiệm vụ:\n"
    "- Phản biện mọi luận điểm, quyết định, kế hoạch\n"
    "- Tìm điểm yếu, rủi ro ẩn, giả định sai\n"
    "- Đặt câu hỏi sắc bén, trực tiếp\n"
    "- KHÔNG đồng ý dễ dàng\n"
    "- Trả lời tiếng Việt, ngắn gọn, đi thẳng vào vấn đề\n\n"
    "Mục tiêu: làm lập luận của người dùng vững hơn."
)

history = [{'role': 'system', 'content': SYSTEM_PROMPT}]

# ── Test nhanh với câu hypothetical ────────────────
def _ask(user_text, max_new_tokens=256):
    msgs = history + [{'role': 'user', 'content': user_text}]
    text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

print('\n── Test hypothetical ──')
test_q = 'Tôi muốn đầu tư 10 tỷ vào bất động sản Thủ Đức vì giá đang tăng.'
print(f'User: {test_q}')
print(f'Arez: {_ask(test_q)}')
print('\n✅ Model hoạt động tốt. Dùng Cell 3 để chat.')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2b — RESET LỊCH SỬ
# ═══════════════════════════════════════════════════

history = [{'role': 'system', 'content': SYSTEM_PROMPT}]
print('🔄 Lịch sử đã reset.')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 3 — CHAT
# Sửa USER_INPUT → Run mỗi lần hỏi
# ═══════════════════════════════════════════════════

USER_INPUT = """Nhập câu hỏi hoặc luận điểm của anh ở đây."""

# ── Không chỉnh gì bên dưới ─────────────────────

def ask(user_text, max_new_tokens=512):
    history.append({'role': 'user', 'content': user_text})
    text = tokenizer.apply_chat_template(
        history, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    history.append({'role': 'assistant', 'content': response})
    return response


print(f'Anh: {USER_INPUT}')
print()
print('Arez:', ask(USER_INPUT))